# ChakraTransformer Kaggle Evaluation & Video Inference

This notebook evaluates the pre-trained ChakraTransformer (Combo 6) on the Kvasir-SEG test set and runs video inference.

**IMPORTANT:** Make sure you have added the following datasets to your Kaggle environment via the **+ Add Input** button in the sidebar:
1. `kvasir-seg` (The Kvasir-SEG dataset)
2. The uploaded zip dataset containing the model code and weights (mounted at `/kaggle/input/datasets/gokulrocky/updated-kaggle` or similar).

In [ ]:
!pip install -q timm opencv-python torchvision

import sys
import os

# Set the path to where you uploaded the kaggle_upload.zip dataset
BASE_DIR = '/BASE_DIR = '/kaggle/input/kaggle-upload-zip4''

if not os.path.exists(BASE_DIR):
    print(f"WARNING: BASE_DIR '{BASE_DIR}' not found.")
    print("Please update BASE_DIR to match your uploaded dataset path.")
else:
    # Copy src to working directory to allow imports
    !cp -r {BASE_DIR}/src /kaggle/working/
    sys.path.append('/kaggle/working/src')
    print("Source code successfully mounted.")


In [ ]:
import torch
from pathlib import Path
import cv2
import numpy as np
from torchvision import transforms
import time
from torch.utils.data import DataLoader

from chakra_transformer.transformer_segmenter import ChakraTransformerSegmenter
from train_pranet import KvasirSEGDataset
from evaluate_all import evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# Load Model
print("Loading ChakraTransformer...")
model = ChakraTransformerSegmenter(backbone_name='vit_large_patch16_384', pretrained=False)

weight_path = f"{BASE_DIR}/chakra_transformer_best.pth"
if not os.path.exists(weight_path):
    # Fallback to check if it's in a weights subfolder
    weight_path = f"{BASE_DIR}/weights/chakra_transformer_best.pth"

model.load_state_dict(torch.load(weight_path, map_location=device))
model.to(device).eval()
print(f"Model weights loaded successfully from: {weight_path}")


## 1. Static Image Evaluation (Kvasir-SEG)
Ensure `kvasir-seg` is in your Kaggle inputs.

In [ ]:
# Dataset Setup
images_dir = Path('/kaggle/input/kvasir-seg/Kvasir-SEG/images')
masks_dir = Path('/kaggle/input/kvasir-seg/Kvasir-SEG/masks')

if images_dir.exists() and masks_dir.exists():
    dataset = KvasirSEGDataset(images_dir, masks_dir, img_size=384, augment=False)
    
    np.random.seed(42)
    all_indices = np.random.permutation(len(dataset)).tolist()

    n_train = int(0.7 * len(dataset))
    n_cal = int(0.15 * len(dataset))
    test_indices = all_indices[n_train + n_cal:]
    test_set = torch.utils.data.Subset(dataset, test_indices)
    test_loader = DataLoader(test_set, batch_size=8, shuffle=False, num_workers=2)
    print(f"Test set size: {len(test_set)}")

    print("Running Evaluation...")
    metrics = evaluate_model(model, test_loader, device)
    print("\n=== FINAL RESULTS ===")
    print(f"Dice Score: {metrics['dice']:.4f}")
    print(f"mIoU Score: {metrics['iou']:.4f}")
    print(f"Precision:  {metrics['precision']:.4f}")
    print(f"Recall:     {metrics['recall']:.4f}")
else:
    print("WARNING: Kvasir-SEG dataset not found. Skipping static evaluation.")


## 2. Video Inference
Run the ChakraTransformer on all `.mp4` and `.avi` videos from the provided directory.

In [ ]:
# Video directory can be the uploaded zip dataset if videos are inside it,
# or a separate dataset path.
video_dir = Path(BASE_DIR) / 'videos'
output_dir = Path('/kaggle/working/output_videos')
output_dir.mkdir(parents=True, exist_ok=True)

video_files = list(video_dir.glob('**/*.mp4')) + list(video_dir.glob('**/*.avi'))
print(f"Found {len(video_files)} videos for testing in {video_dir}.")

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
])

for vid_path in video_files:
    print(f"\nProcessing {vid_path.name}...")
    cap = cv2.VideoCapture(str(vid_path))
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    
    out_path = str(output_dir / f"annotated_{vid_path.name}")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, int(fps), (width, height))
    
    frame_count = 0
    start_time = time.time()
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_tensor = transform(rgb_frame).unsqueeze(0).to(device)
        
        with torch.no_grad():
            outputs = model(input_tensor)
            if isinstance(outputs, (tuple, list)):
                pred = outputs[0]
            else:
                pred = outputs
            pred = torch.sigmoid(pred).squeeze().cpu().numpy()
            
        pred_resized = cv2.resize(pred, (width, height))
        mask = (pred_resized > 0.5).astype(np.uint8) * 255
        
        colored_mask = np.zeros_like(frame)
        colored_mask[:, :, 1] = mask
        
        overlay = cv2.addWeighted(frame, 0.7, colored_mask, 0.3, 0)
        out.write(overlay)
        
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"  Processed {frame_count} frames...")
            
    cap.release()
    out.release()
    
    if frame_count > 0:
        elapsed = time.time() - start_time
        print(f"Finished {vid_path.name} - {frame_count} frames in {elapsed:.2f}s ({frame_count/elapsed:.1f} FPS)")
        print(f"Saved to {out_path}")
    else:
        print(f"Failed to process any frames from {vid_path.name}.")
